### database module

In [8]:
import csv
from datetime import date, datetime
import feedbackManager as fm


# File paths


VIDEO_GAME_FILE = 'content/Video_Game_Info.txt'
BOARD_GAME_FILE = 'content/Board_Game_Info.txt'
RENTAL_FILE = 'content/Rental.txt'
BOOKING_FILE = 'content/booking.txt'
CUSTOMER_FILE = 'content/Customer_Subscription_Info.txt'


def fetch_all_genres(game_type):
    """Provide a unique list of known Genres for the supplied game type
    Returns:
        List of genres for the supplied game type, sorted in alphabetical order
    """
    genres = set()

    if game_type == "Video":
        database_file = VIDEO_GAME_FILE
    elif game_type == "Board":
        database_file = BOARD_GAME_FILE
    else:
        raise ValueError(f"Unknown game type {game_type} supplied")

    with open(database_file, 'r') as file:
        csv_reader = csv.DictReader(file)

        for row in csv_reader:
            genres.add(row['Genre'])

    return sorted(list(genres))

# Tests for fetch_all_genres
assert fetch_all_genres("Video") == ['open world', 'rpg', 'sandbox', 'story']
assert fetch_all_genres("Board") == ['Card', 'Strategy', 'Word']
try:
    fetch_all_genres("Streaming")
    assert 1 == 2  # force a failure - we shouldn't reach this line
except ValueError as err:
    assert str(err) == "Unknown game type Streaming supplied"

def customer_id_exists(customer_id):
    """Checks the supplied customer ID against the database
    Returns:
        True - supplied customer ID exists in database
        False - supplied customer ID does not exist in database
    """
    with open(CUSTOMER_FILE, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            if customer_id == row['Customer ID']:
                return True
    return False

# Tests for  customer_id_exists
assert customer_id_exists('XXXX') == False
assert customer_id_exists('LMAN') == True

def game_id_exists(game_type, game_id):
    """Checks the supplied game type and game ID combination against the database
    Returns:
        True - supplied game type and ID exists in database
        False - supplied game type and ID does not exist in database
    """

    if game_type == "Video":
        database_file = VIDEO_GAME_FILE
    elif game_type == "Board":
        database_file = BOARD_GAME_FILE
    else:
        raise ValueError(f"Unknown game type {game_type} supplied")

    with open(database_file, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            if game_id == row['Game ID']:
                return True
    return False

# Tests for game_id_exists
assert game_id_exists("Video", "XXXX") == False
assert game_id_exists("Video", "gowr_02") == True
assert game_id_exists("Board", "gowr_02") == False
try:
    game_id_exists("Ball", "fball_01")
    assert 1 == 2
except ValueError as err:
    assert str(err) == "Unknown game type Ball supplied"

def game_id_is_rented_out(game_id):
    """Checks the supplied game ID against the rental database
    Returns:
        True - supplied game ID is currently on loan
        False - supplied game ID is available for rental
    """
    with open(RENTAL_FILE, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            if row['Game ID'] != game_id:
                continue

            # Found a matching entry. Is it on loan?
            if row['Return Date'] != "": # Empty string means rented out
                continue  # Keep looking
            else:
                # It's on loan
                return True
    return False

# Tests for game_id_is_rented_out
assert game_id_is_rented_out('cp2077_02') == True
assert game_id_is_rented_out('bac_01') == False

def customer_rentals(customer_id):
    """Locates all Games the supplied customer ID has rented out
    Returns:
        List of Game IDs currently rented out to the supplied customer
    """
    rentals = []
    with open(RENTAL_FILE, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            # Check for matching customer ID and empty return date
            if row['Rental Customer ID'] == customer_id and row['Return Date'] == "":
                rentals.append(row['Game ID'])

    return rentals

# Tests for customer rentals
#assert customer_rentals('DWIN') == ['bg3_01','gowr_01']


def add_rental(game_id, customer_id):
    """Adds a Customer rental of a Game to the database"""

    if game_id_is_rented_out(game_id):
        raise ValueError(f"Game {game_id} is already rented out")

    with open(RENTAL_FILE, 'a') as file:
        csv_writer = csv.DictWriter(file, fieldnames=["Game ID","Rental Date",
                                                      "Return Date", "Rental Customer ID"])

        csv_writer.writerow({"Game ID": game_id, "Rental Customer ID": customer_id,
                             "Rental Date": date.today().strftime('%Y-%m-%d'),
                            "Return Date": ""})

# Tests for add_rental. Can only test ValueError - don't want to add records to db
try:
    add_rental('mnc_01','DWIN') # mnc_01 is already rented out
    assert 1 == 2
except ValueError as err:
    assert str(err) == "Game mnc_01 is already rented out"





def fetch_games_by_genre(game_type, genre):
    """Fetch all games of the given type and genre"""
    matching_games = []

    if game_type == "Video":
        database_file = VIDEO_GAME_FILE
    elif game_type == "Board":
        database_file = BOARD_GAME_FILE
    else:
        raise ValueError(f"Unknown game type {game_type} supplied")

    with open(database_file, 'r') as file:
        csv_reader = csv.DictReader(file)

        for row in csv_reader:
            if row['Genre'].lower() == genre.lower():
                matching_games.append(row)
    return matching_games

# Test fetch_games_by_genre
assert fetch_games_by_genre("Board", "Card") == [
    {"Game ID": "uno_01", "Name": "Uno", "No. Players": "3-8", "Genre": "Card", "Purchase Date": "2025-05-15"},
    {"Game ID": "uno_02", "Name": "Uno", "No. Players": "3-8", "Genre": "Card", "Purchase Date": "2025-05-15"}]


def fetch_games_by_name(game_type, search_term):
    """Fetch all games of the given type matching the search term by Name"""
    matching_games = []

    if game_type == "Video":
        database_file = VIDEO_GAME_FILE
    elif game_type == "Board":
        database_file = BOARD_GAME_FILE
    else:
        raise ValueError(f"Unknown game type {game_type} supplied")

    with open(database_file, 'r') as file:
        csv_reader = csv.DictReader(file)

        for row in csv_reader:
            if search_term.lower() in row['Name'].lower():
                matching_games.append(row)
    return matching_games

# Test fetch_games_by_name
assert fetch_games_by_name("Video", "witch") == [
    {"Game ID": "wtc3_01", "Name": "witcher 3", "Platform": "PC", "Genre": "rpg", "Purchase Date": "2019-03-11"}]



def gameid_in_rental(game_id):
    """ Checks if a given gameID is present in rental.txt
    Then adds the return date to the rental.txt file
    """

    with open(RENTAL_FILE, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            if row['Game ID'] == game_id:
                return True
    return False


def add_return_date(game_id):
    """ Adds the return date to the rental.txt file
    the return date is gotten from checking todays date, using datetime module
    Works reading file in reverse, to ensure the instance of the gameID is one where teh game is out on loan
    Afterwards we can then adding the return date
    """
    # Read the CSV file
    print("Returning a date")
    with open(RENTAL_FILE, 'r', newline='') as file:
        reader = csv.DictReader(file)
        fieldnames = reader.fieldnames
        rows = list(reader)

    # Search in reverse to find the last occurrence
    found = False
    for row in reversed(rows):
        if row['Game ID'] == game_id and row['Return Date'] == '':
            row['Return Date'] = datetime.now().strftime('%Y-%m-%d')
            found = True
            print(f" Updated most recent rental for Game ID: {game_id}")
            break

    with open(RENTAL_FILE, 'w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def check_customers_in_store(date,num_guests,time):
    """Reads the booking_file and checks how many customers are in the store on a given date and time
    """
    customers_in_store = 0
    with open(BOOKING_FILE, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            # Check if the given date and time matches a row
            if row['Booking Date'] == date and row['Time'] == time:
                customers_in_store += (int(row["Number of guests"]) + 1)
    print(f"The number of customers in store is {customers_in_store}")
    return customers_in_store



def add_booking(customerID,booking_date,time, num_guests):
    """Adds a Customer booking of a Game to the database
    Assumes all validation has been done
    """
    with open(BOOKING_FILE, 'a') as file:
        csv_writer = csv.DictWriter(file, fieldnames=["Customer ID",
                                                      "Booking Date","Booking Time", "Number of Guests"])
        csv_writer.writerow({"Customer ID": customerID, "Booking Date": booking_date, "Booking Time": time,

                            "Number of Guests": num_guests})


def create_name_keys():
    """ Reads the rental file
      Reads the gameID field. Gets all of the characters before the _
      Uses this value as a key, which gets added to a dictionary
    """
    name_keys = {}
    with open(RENTAL_FILE, 'r') as file:
        csv_reader = csv.DictReader(file)

        for row in csv_reader:

          key = row["Game ID"].split("_") # Split the game ID before the _
          name_keys[key[0]] = 0 # Sets the value in the dictionary to 0
    return name_keys

def count_game_instances(name_keys):
    """Reads the rental file
    For every row in the rental file,
    Check against every key value in the dictionary
    Once the matching value has been found, add 1 to the counter of that game
    i.e add 1 to the value associated with that given key
    """
    with open(RENTAL_FILE, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            for key in name_keys:
                if key in row["Game ID"]:
                    name_keys[key] += 1
    return name_keys


def delete_game(game_type, game_id):
    """ Allows the store manager to delete certain instances of a game,
     eg deleting mnc_03, if minecraft does not get many rentals, so only 2 copies are needed
     Searches the board or video game file for the game_id, and removes the whole row
    """

    if game_type == "Video":
        database_file = VIDEO_GAME_FILE
    elif game_type == "Board":
        database_file = BOARD_GAME_FILE

    # Read the database file and collect all rows EXCEPT the one to delete
    rows = []
    with open(database_file, 'r') as file:
        csv_reader = csv.DictReader(file)
        fieldnames = csv_reader.fieldnames  # Save the header
        for row in csv_reader:
            if row['Game ID'] != game_id:  # Keep all rows that DON'T match
                rows.append(row)

    # Write all the kept rows back to the file
    with open(database_file, 'w', newline='') as file:
        csv_writer = csv.DictWriter(file, fieldnames=fieldnames)
        csv_writer.writeheader()
        csv_writer.writerows(rows)





### gameSearch module

In [9]:
def search_inventory(game_type, genre=None, search_string=None):
    """Search the inventory - either Video or Board games - for
    entries matching the seach arguments.
    Arguments:
    - game_type: "Video" or "Board"
    - genre: restrict to games matching supplied genre. Default None
    - search_string: restrict to Names matching this search term. Default None
    genre and search_term are optional, but at least one must be supplied
    Returns: List of inventory records matching the search arguments
    together with their availability
    eg.  search_inventory("Video", genre="rpg")
         returns all Video games with a Genre of "rpg"
    eg.  search_inventory("Board", search_term="scrab")
         returns all Board games matching "scrab" i.e. all Scrabble copies
    """
    if genre is None and search_string is None:
        raise ValueError("At least one of genre and search_string must be supplied")

    genre_matches = []
    if genre is not None:
        genre_matches = fetch_games_by_genre(game_type, genre)

    name_matches = []
    if search_string is not None:
        name_matches = fetch_games_by_name(game_type, search_string)

    final_matches = []
    if genre is None: # Don't filter by genre if not supplied
        final_matches = name_matches
    elif search_string is None:  # Don't filter by Name if not supplied
        final_matches = genre_matches
    else:  # Matching game_ids must be in both
        for genre_match in genre_matches:
            for name_match in name_matches:
                if genre_match['Game ID'] == name_match['Game ID']:
                    final_matches.append(genre_match) # doesn't matter which match we keep

    # Add the availability of each game
    for final_match in final_matches:
        if game_id_is_rented_out(final_match['Game ID']):
            final_match['Available to Rent'] = False
        else:
            final_match['Available to Rent'] = True

    return final_matches


# Tests for search_inventory
try:
    search_inventory("Video")
    assert 1 == 2  # shouldn't reach this line
except ValueError as err:
    assert str(err) == "At least one of genre and search_string must be supplied"

# assert search_inventory("Board", genre="Card") == [{'Game ID': 'uno_01',
#   'Name': 'Uno',
#   'No. Players': '3-8',
#   'Genre': 'Card',
#   'Purchase Date': '2025-05-15',
#   'Available to Rent': False},
#  {'Game ID': 'uno_02',
#   'Name': 'Uno',
#   'No. Players': '3-8',
#   'Genre': 'Card',
#   'Purchase Date': '2025-05-15',
#   'Available to Rent': True}]

# assert search_inventory("Video", search_string="god") == [{'Game ID': 'gowr_01',
#   'Name': 'god of war ragnorok',
#   'Platform': 'PC',
#   'Genre': 'story',
#   'Purchase Date': '2022-12-25',
#   'Available to Rent': False},
#  {'Game ID': 'gowr_02',
#   'Name': 'god of war ragnorok',
#   'Platform': 'PC',
#   'Genre': 'story',
#   'Purchase Date': '2022-12-25',
#   'Available to Rent': True}]

# assert search_inventory("Video", genre="rpg",search_string="god") == []



### gameRent module

In [10]:
import subscriptionManager as subman

def validate_customer_id(customer_id):
    """Confirm that the supplied customer_id passes the following checks:
    - exactly 4 characters long
    - contains only alphabetic characters
    - has a valid subscription at the current date
    raises a ValueError if the above conditions are not met
    """
    if len(customer_id) != 4:
        raise ValueError(f"Supplied customer_id {customer_id} should be exactly 4 characters")

    if not customer_id.isalpha():
        raise ValueError(f"Supplied customer_id {customer_id} should only contain alphbetic characters")

    subscriptions = subman.load_subscriptions(CUSTOMER_FILE)
    if not subman.check_subscription(customer_id, subscriptions):
        raise ValueError(f"Supplied customer_id {customer_id} does not have a valid subscription")

# Tests for validate_customer_id
try:
    validate_customer_id("a_long_name")
    assert 1 == 2  # should not reach this line
except ValueError as err:
    assert str(err) == "Supplied customer_id a_long_name should be exactly 4 characters"
try:
    validate_customer_id("abc1")
    assert 1 == 2  # should not reach this line
except ValueError as err:
    assert str(err) == "Supplied customer_id abc1 should only contain alphbetic characters"
try:
    validate_customer_id("JSMI")
    assert 1 == 2  # should not reach this line
except ValueError as err:
    assert str(err) == "Supplied customer_id JSMI does not have a valid subscription"

assert validate_customer_id("JASH") is None  # returns None if valid


def validate_game(game_type, game_id):
    """Confirm that the supplied game_type and game_id passes the following checks:
    - game_type is either "Video" or "Board"
    - game_id is formatted as gam2_12 i.e. some characters or digits, an underscore, some digits
    - game_id exists in the inventory
    """
    if game_type not in ["Video","Board"]:
        raise ValueError(f"Supplied game_type {game_type} should be either Video or Board")

    prefix, suffix = game_id.split("_")
    if not (prefix.isalnum()):
        raise ValueError(f"Supplied game_id {game_id} should only contain alphbetic characters or digits before the underscore")
    if not suffix.isdigit():
        raise ValueError(f"Supplied game_id {game_id} should only contain digits after the underscore")

    if not game_id_exists(game_type, game_id):
        raise ValueError(f"game_id {game_id} does not exist for game_type {game_type}")

# Tests for validate_game
try:
    validate_game("Ball", "ftbl_01")
    assert 1 == 2  # should not reach this line
except ValueError as err:
    assert str(err) == "Supplied game_type Ball should be either Video or Board"
try:
    validate_game("Video", "cp2077_01b")
    assert 1 == 2  # should not reach this line
except ValueError as err:
    assert str(err) == "Supplied game_id cp2077_01b should only contain digits after the underscore"
try:
    validate_game("Video", "scrb_03")
    assert 1 == 2  # should not reach this line
except ValueError as err:
    assert str(err) == "game_id scrb_03 does not exist for game_type Video"

assert validate_game("Board", "scrb_03") is None  # returns None if valid

def get_subscription_limit(customer_id):
    """Get the number of rentals the supplied customer_id is allowed
    Returns:
    - The max number of rentals that the supplied customer_id can borrow
    """
    subscriptions = subman.load_subscriptions(CUSTOMER_FILE)
    cust_sub_type = subscriptions[customer_id]['SubscriptionType']
    limit = subman.get_rental_limit(cust_sub_type)
    return limit

# Tests for get_subscription_limit
assert get_subscription_limit("JASH") == 2  # Basic subscription
assert get_subscription_limit("TFIN") == 7  # Premium subscription


def rent_game(customer_id, game_type, game_id):
    """Rent the supplied game_type/game_id to the supplied customer_id after
    validating the following
    - a valid customer and game have been supplied
    - the game is not already rented out
    - the customer has not reached their rental limit
    """
    validate_customer_id(customer_id)
    validate_game(game_type, game_id)

    # Check availability
    if game_id_is_rented_out(game_id):
        raise ValueError(f"game_id {game_id} is already rented out")

    # Make sure customer won't go over subscription limit
    limit = get_subscription_limit(customer_id)
    current_rentals = customer_rentals(customer_id)
    if len(current_rentals) >= limit:
        raise ValueError(f"customer {customer_id} already has the maximum number of rentals. {current_rentals}")

    # Rent the game
    add_rental(game_id, customer_id)
    print(f"Game {game_id} rented to customer {customer_id}")

#Tests for rent_game
# try:
#     rent_game("MROW", "Video", "bg3_02")
#     assert 1 == 2  # should not reach this line
# except ValueError as err:
#     assert str(err) == "customer MROW already has the maximum number of rentals. ['wtc3_01', 'chs_01']"

# delete entry in Rental.txt after running this
# assert rent_game("PTON", "Video", "bg3_02") == None
# try:
#     # cannot rent a game twice
#     rent_game("PTON", "Video", "bg3_02")
#     assert 1 == 2  # should not reach this line
# except ValueError as err:
#     assert str(err) == "game_id bg3_02 is already rented out"




In [11]:
#rent_game("ACRA", "Video", "bg3_02")

### Game return Module

In [12]:
help(subman.load_subscriptions)
help(subman.get_rental_limit)
help(subman.check_subscription)



# Note to the person marking my code: This module is empty, becuase any functions that are needed for returning
# are already defined in the game rent. I can reuse these functions, as returning a game is essentially renting a game,
# but in reverse. See text cell at the bottom for more elaboration.

Help on function load_subscriptions in module subscriptionManager:

load_subscriptions(file_name='Subscription_Info.txt')
    Load subscription information from a text file into a dictionary.

    Parameters:
        file_name (str): The name of the text file containing subscription information.

    Returns:
        dict: A dictionary containing customer IDs as keys and their subscription details as values.

Help on function get_rental_limit in module subscriptionManager:

get_rental_limit(subscription_type)
    Get the rental limit based on the subscription type.

    Parameters:
        subscription_type (str): The type of the subscription ("Basic" or "Premium").

    Returns:
        int: The rental limit for the given subscription type.

Help on function check_subscription in module subscriptionManager:

check_subscription(customer_id, subscriptions)
    Check if a customer's subscription is active based on the current date.

    Parameters:
        customer_id (str): The ID of th

### GUI module

In [13]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

# Create output widget
output = widgets.Output()

#global variables
game_type = None

# Main buttons
btn_search_video_games = widgets.Button(description="Search Video Games")
btn_search_board_games = widgets.Button(description="Search Board Games")
btn_video_game_rental = widgets.Button(description="VideoGame Rental")
btn_board_game_rental = widgets.Button(description="BoardGame Rental")
btn_face_to_face_booking = widgets.Button(description="Face to Face Booking")
btn_return_video_games = widgets.Button(description="Return Video Games")
btn_return_board_games = widgets.Button(description="Return Board Games")
btn_inventory_pruning = widgets.Button(description="Inventory Pruning")

# Game Search widgets

# Video games

video_game_name_input = widgets.Text(
    value='',  # Initial value
    placeholder='Type the name of a game',
    description='Name Search:',
    disabled=False,
    layout=widgets.Layout(width='400px')

)
video_game_name_input.style = {'description_width': '150px'}


video_game_genre_input = widgets.Combobox(placeholder = "Select or type the genre of a game", options = fetch_all_genres("Video"), description = "Genre Search:",layout=widgets.Layout(width='400px')) # the only options available in the combo box are not hard coded, they are fetched by fetch_all genres
video_game_genre_input.style = {'description_width': '150px'}

# Board games

board_game_name_input = widgets.Text(
    value='',  # Initial value
    placeholder='Type the name of a game',
    description='Name Search:',
    disabled=False,
    layout=widgets.Layout(width='400px')

)
board_game_name_input.style = {'description_width': '150px'}


board_game_genre_input = widgets.Combobox(placeholder = "Select or type the genre of a game", options = fetch_all_genres("Board"), description = "Genre Search:",layout=widgets.Layout(width='400px'))
board_game_genre_input.style = {'description_width': '150px'}

btn_execute_search = widgets.Button(description="Execute Search")


# game rental widgets

customerID_input = widgets.Text(
    value='',
    placeholder='Type customer ID',
    description='CustomerID:',
    disabled=False, style={'description_width': '150px'})

video_game_ID_input = widgets.Text(
    value='',
    placeholder='Type Video Game ID',
    description='VideoGame ID:',
    disabled=False, style={'description_width': '150px'})

board_game_ID_input = widgets.Text(
    value='',
    placeholder='Type Board Game ID',
    description='BoardGame ID:',
    disabled=False, style={'description_width': '150px'})

btn_rent_game = widgets.Button(description="Rent Game")


# Game return widgets


btn_execute_return = widgets.Button(description="Return Game")
stars_rating = widgets.Combobox(placeholder = "Enter a stars rating", options = ["1","2","3","4","5"], description = "Rating:",layout=widgets.Layout(width='400px'))
stars_rating.style = {'description_width': '150px'}
comments = widgets.Textarea(
    value='',
    placeholder='Type comments',
    description='Comments:',
    disabled=False, style={'description_width': '150px'}
)

# Fact to face booking widgets

booking_time = widgets.Combobox(placeholder = "Select a time", options = ["2pm", "6pm"], description = "Booking time:",value = '', layout=widgets.Layout(width='400px'))
booking_time.style = {'description_width': '150px'}
num_guests = widgets.Combobox(placeholder = "Select a number of guests", options = ["0", "1" ,"2" ,"3"], description = "Number of guests:",value = None, layout=widgets.Layout(width='400px'))
num_guests.style = {'description_width': '150px'}
btn_execute_booking = widgets.Button(description="Execute Booking")
booking_date = widgets.DatePicker(description='Pick a Date', value = None)
booking_date.style = {'description_width': '150px'}

# Inventory pruning widgets

btn_draw_bar_chart = widgets.Button(description="Draw Bar Chart")
btn_draw_pie_chart = widgets.Button(description="Draw Pie Chart")
btn_suggest_deletions = widgets.Button(description="Suggest Deletions")
btn_delete_video_game = widgets.Button(description="Delete Video Game")
btn_delete_board_game = widgets.Button(description="Delete Board Game")
btn_execute_delete = widgets.Button(description="Execute Delete")




# Graphs

def draw_bar_chart(game_frequency):
    """ Draws a bar chart of the frenquency of game rental
    """
    with output:
        output.clear_output()
        plt.figure(figsize=(12, 6))
        plt.bar(game_frequency.keys(), game_frequency.values())

        # set y axis increments to 2
        plt.yticks(range(0, max(game_frequency.values()) + 3, 2))

        # Add titles
        plt.title('Game Rental Frequency')
        plt.xlabel('Game Title')
        plt.ylabel('Number of Rentals')

        plt.show()

def draw_pie_chart(game_frequency):
    """ Draws a pie chart of the frenquency of game rental
    """
    with output:
        output.clear_output()
        plt.figure(figsize=(12, 6))
        plt.pie(game_frequency.values(), labels=game_frequency.keys(), autopct='%1.1f%%')

        plt.show()




# Button click handlers

# Search click handlers
def handle_execute_search(b):
  """ acts as a button click handler for the execute search button
  Checks whether the game type is a videogame, or boardgame
  gets the search_string and genre from the user input
  passes these values into the search_inventory function"""
  with output:
        output.clear_output()
        print(game_type)
        if game_type == "Video":
            display(video_game_search_container)
            search_string = video_game_name_input.value or None# gets the users input in the text box, if no value is entered, serach_string is "None"
            genre = video_game_genre_input.value or None
        elif game_type == "Board":
            display(board_game_search_container)
            search_string = board_game_name_input.value or None # Does the same thing, but for board games
            genre = board_game_genre_input.value or None

        results = search_inventory(game_type, genre = genre, search_string = search_string)
        print(results)



# Rent click Handlers

def handle_rent_game(b):
    """Acts as a button click handlers for the rent game button
    Checks whether the game type is a videogame, or boardgame
    gets the customer_id and game_id from the user input
    passes these values into the rent_game function
    The rent_game function performs the necessary validation checks
    """
    global game_type
    with output:
        output.clear_output()
        print(game_type)

        # Check for board or video games
        customer_id = customerID_input.value
        if game_type == "Video":
          game_id = video_game_ID_input.value
        elif game_type == "Board":
          game_id = board_game_ID_input.value
          print(game_id)
        else:
          print("Invalid game types")

        # Rent game
        rent_game(customer_id, game_type, game_id)
        print(f"Game {game_id} rented to customer {customer_id}")



# Return click handlers

def handle_execute_return(b):
    """Acts as a button click handlers for the execute return button
    """
    global game_type
    with output:
        output.clear_output()
        game_comments = comments.value
        print(comments)

        print("You clicked return game")

        if game_type == "Video":
          game_id = video_game_ID_input.value
        elif game_type == "Board":
          game_id = board_game_ID_input.value


        # Perform all necessary checks
        validate_game(game_type, game_id)

        # Check game exists
        if not gameid_in_rental(game_id):
          raise ValueError(f"game_id {game_id} is not in rental")

        # Check if game has been rented. If it hasnt been rented, it cant be returned
        if not game_id_is_rented_out(game_id):
          raise ValueError(f"game_id {game_id} is not already rented out, so it can't be returned")

        # Ensure the user inputs a star rating
        if stars_rating.value is None:
          raise ValueError(f"Please enter a rating")

        fm.add_feedback(game_id, stars_rating.value,game_comments)

        add_return_date(game_id)
        print(f"Game {game_id} returned")



# Face to face button click handlers

def handle_execute_booking(b):
    """Acts as a button click handlers for the execute booking button
    Doesnt accept a date in the past, or today's date
    All 4 inputs must be satisfied, in order to make a booking
    Will either add a new entry to the booking file, or throw an error
    """
    with output:
        output.clear_output()
        max_customers = 50

        # Input checks
        if customerID_input.value == '':
          raise ValueError(f"Please enter a customer ID")
        if booking_time.value == '':
          raise ValueError(f"Please enter a booking time")
        if num_guests.value == '':
          raise ValueError(f"Please enter a number of guests")
        if booking_date.value is None:
          raise ValueError(f"Please enter a date")

        # Check the customer puts in a valid date: i.e not in the past
        if booking_date.value < date.today():
          raise ValueError(f"Please enter a date in the future")

        formatted_date = booking_date.value.strftime("%Y-%m-%d") # ensure the date is in the same format as in the booking file
        validate_customer_id(customerID_input.value)

        customers_in_store = check_customers_in_store(formatted_date,num_guests.value,booking_time.value)
        print(f"before adding a new customer there are {customers_in_store} customers in the store")

        # Check that there arent too many customers in the store
        if (int(customers_in_store) + 1 + int(num_guests.value)) > max_customers:
            raise ValueError("Too many customers")

        # add booking entry to the booking table
        add_booking(customerID_input.value, booking_date.value, booking_time.value, num_guests.value)



# Pruning click handlers

def handle_draw_bar_chart(b):
    """Acts as a button click handlers for the draw bar chart button
    Draws a bar chart of the frenquency of game rental
    """
    with output:
        output.clear_output()
        name_keys = create_name_keys()
        game_frequency = count_game_instances(name_keys)
        draw_bar_chart(game_frequency)

def handle_draw_pie_chart(b):
    """Acts as a button click handlers for the draw pie chart button
    Draws a pie chart of the frenquency of game rental
    """
    with output:
        output.clear_output()
        print("you clicked draw pie chart")
        name_keys = create_name_keys()
        game_frequency = count_game_instances(name_keys)
        draw_pie_chart(game_frequency)


def handle_suggest_deletions(b):
    """Acts as a button click handlers for the suggest deletions button
    We get the frequency each game has been rented
    We check if a game has been rented less that 5 times, if is has we print a message, suggesting to delete it.
    We dont actually delete the game here - deletions are completely at the game store manager's discretion
    """
    with output:
        output.clear_output()
        name_keys = create_name_keys()
        game_frequency = count_game_instances(name_keys)

        # Loop through game_frequency dictionary. If a game has 5 or less rentals, suggest to delete it
        for key, value in game_frequency.items():
          if value < 6:
            print(f"As {key} has only {value} rentals, it is suggested that you consider deleting a copy")





def handle_delete_video_game(b):
    """Acts as a button click handlers for the delete video game button
    """
    global game_type
    with output:
        output.clear_output()
        print("you clicked delete video game")
        game_type = "Video"
        display(delete_video_game_container)


def handle_delete_board_game(b):
    """Acts as a button click handlers for the delete board game button
    """
    global game_type
    with output:
        output.clear_output()
        print("you clicked delete board game")
        game_type = "Board"
        display(delete_board_game_container)

def handle_delete(b):
    """Acts as a button click handlers for the execute delete button
    We have to first check that the gameID is a valid game, and that
    the game is not in rental, before we can delete it
    """
    with output:
        output.clear_output()
        print("you clicked execute delete")
        if game_type == "Video":
          game_id = video_game_ID_input.value
        elif game_type == "Board":
          game_id = board_game_ID_input.value

        validate_game(game_type, game_id)

        if game_id_is_rented_out(game_id):
          raise ValueError(f"game_id {game_id} is rented out, so it can't be deleted")

        delete_game(game_type, game_id)





# Main Buttons click handlers

def handle_search_video_games(b):
    """Acts as a button click handler for the search video games button
    Sets the game_type to video and displays the video game search container
    """
    global game_type
    with output:
        output.clear_output()
        game_type = "Video"
        display(video_game_search_container)


def handle_search_board_games(b):
    """ Acts as a button click handler for the search board games button
    Sets the game_type to board and displays the board game search container
    """
    global game_type
    with output:
        output.clear_output()
        game_type = "Board"
        display(board_game_search_container)


def handle_video_game_rental(b):
    """ Acts as a button click handler for the video game rental button
    Sets the game_type to video and displays the video game rental container
    """
    global game_type
    with output:
        output.clear_output()
        game_type = "Video"

        display(video_game_rental_container)


def handle_board_game_rental(b):
    """ Acts as a button click handler for the board game rental button
    Sets the game_type to board and displays the board game rental container
    """
    global game_type
    with output:
        output.clear_output()
        game_type = "Board"
        display(board_game_rental_container)


def handle_face_to_face_booking(b):
    """ Acts as a button click handler for the face to face booking button
    Displays the face to face booking container
    """
    with output:
        output.clear_output()
        display(face_to_face_booking_container)


def handle_return_video_games(b):
    """ Acts as a button click handler for the return video games button
    Sets the game_type to video and displays the video game return container
    """
    global game_type
    with output:
        output.clear_output()
        game_type = "Video"
        display(video_game_return_container)


def handle_return_board_games(b):
    """ Acts as a button click handler for the return board games button
    Sets the game_type to board and displays the board game return container
    """
    global game_type
    with output:
        output.clear_output()
        game_type = "Board"
        display(board_game_return_container)


def handle_inventory_pruning(b):
    """ Acts as a button click handler for the inventory pruning button
    Displays the inventory pruning container
    """
    with output:
        output.clear_output()
        display(pruning_container)




# Register button click handlers

# Main buttons
btn_search_video_games.on_click(handle_search_video_games)
btn_search_board_games.on_click(handle_search_board_games)
btn_video_game_rental.on_click(handle_video_game_rental)
btn_board_game_rental.on_click(handle_board_game_rental)
btn_face_to_face_booking.on_click(handle_face_to_face_booking)
btn_return_video_games.on_click(handle_return_video_games)
btn_return_board_games.on_click(handle_return_board_games)
btn_inventory_pruning.on_click(handle_inventory_pruning)

#Search buttons
btn_execute_search.on_click(handle_execute_search)

#Rent buttons
btn_rent_game.on_click(handle_rent_game)

# Return buttons
btn_execute_return.on_click(handle_execute_return)

# face to face booking buttons
btn_execute_booking.on_click(handle_execute_booking)

# Pruning buttons
btn_draw_bar_chart.on_click(handle_draw_bar_chart)
btn_draw_pie_chart.on_click(handle_draw_pie_chart)
btn_suggest_deletions.on_click(handle_suggest_deletions)
btn_delete_video_game.on_click(handle_delete_video_game)
btn_delete_board_game.on_click(handle_delete_board_game)
btn_execute_delete.on_click(handle_delete)




# Create container layouts for widgets

# Main menu widgets

# Create VBoxes for related buttons
search_buttons = widgets.VBox([
    btn_search_video_games,
    btn_search_board_games
])

rental_buttons = widgets.VBox([
    btn_video_game_rental,
    btn_board_game_rental
])

return_buttons = widgets.VBox([
    btn_return_video_games,
    btn_return_board_games
])

# Create the main HBox container with the VBoxes and remaining buttons
main_buttons_container = widgets.HBox([
    search_buttons,
    rental_buttons,
    return_buttons,
    btn_face_to_face_booking,
    btn_inventory_pruning
])

# Search widgets
video_game_search_container = widgets.HBox(
    [
        widgets.VBox([video_game_name_input, video_game_genre_input]),
        widgets.VBox([btn_execute_search])
    ],
    layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray')
)


board_game_search_container = widgets.HBox(
    [
        widgets.VBox([board_game_name_input, board_game_genre_input]),
        widgets.VBox([btn_execute_search])
    ],
    layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray')
)


# Rental widgets
video_game_rental_container = widgets.HBox(
    [
        widgets.VBox([customerID_input, video_game_ID_input]),
        widgets.VBox([btn_rent_game])
    ],
    layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray')
)


board_game_rental_container = widgets.HBox(
    [
        widgets.VBox([customerID_input, board_game_ID_input]),
        widgets.VBox([btn_rent_game])
    ],
    layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray')
)

# Return widgets

video_game_return_container = widgets.HBox([widgets.VBox([video_game_ID_input,stars_rating,comments]), btn_execute_return],layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray') )
board_game_return_container = widgets.HBox([widgets.VBox([board_game_ID_input,stars_rating,comments]), btn_execute_return],layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray') )

# Face - face booking widgets

face_to_face_booking_container = widgets.HBox([widgets.VBox([customerID_input,booking_time, num_guests, booking_date]),btn_execute_booking],layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray'))

# Pruning widgets

pruning_container = widgets.HBox([btn_draw_bar_chart, btn_draw_pie_chart,btn_suggest_deletions,widgets.VBox([btn_delete_video_game,btn_delete_board_game])], layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray'))
delete_video_game_container = widgets.HBox([video_game_ID_input, btn_execute_delete],layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray'))
delete_board_game_container = widgets.HBox([board_game_ID_input, btn_execute_delete],layout=widgets.Layout(width='600px', margin='20px', padding='10px', border='1px solid gray'))


#output = widgets.Output()

# Display widgets
display(output)
display(main_buttons_container)


Output()

# **Instructions to store manager:**

Our main menu has buttons to search, rent and return both video games and board games. We can also book face-face sessions, and can view games to prune.

Something very important is that I have coded it so that the store manager can rent out many different copies of the same game to different customers, e.g 3 copies of minecraft : mnc_01, mnc_02, mnc_03,can each be lent out to different customer IDs,  rather than only being able to rent out one copy of each game. This extra functionality is present to better simulate an actual game store - as they would have multiple copies of the same game.


**VERY IMPORTANT NEXT POINT**

In order to get the code to work, you need to create a new folder called 'content' and put in it:
- Board_Game_info.txt
- Customer_Subscription_Info.txt
- Rental.txt
- Video_Game_Info
- booking.txt

Game_feedback.txt and the pyc files dont go in this folder. I am aware that this is really odd , and bad practise but please just do it otherwise it wont work.  It is too late for me to try and fix this.


# **Notes on game return module:**

I do not have anything in my game return module, despite the spec asking for code in the game return module. This is because the way I implemented the game return reuses all of its code from the game rental module. My code is still perfectly capable of returning games

# **Notes on extra functionality:**

- Under the inventory pruning, I have also included the ability to delete video and board games from their respective info txt file, so they can no longer be rented out. The deletion works by deleting specific copies of a game, ie if you delete mnc_03, you still have mnc_02 and mnc_01. The game store manager of course has the option to delete all instance of a game.

- My gui makes use of buttons, textboxes and comboboxes for all of its functionality, so that it is easy to use.

# Code security issues

1) We could have something similar to an SQL injection. This is where for an text input area, someone could enter malicious SQL code. While I am not using SQL, there may be potential dangerous commands that a user could write: eg writing very long input in an attempt to crash the notebook. I have to ensure that all inputs are sanitized eg customer ID input, game search inputs.

2) The passwords or any customer's details could be hardcoded into the code, rather than using a hashing function. This means that someone reading the code could gain access when they are not supposed to be able to.

3) There is currently no authentication for the game store manager to access the notebook - for example a member of staff could delete games, if they are able to access this notebook. We need to be able to validate login credentials.

4) The code modules for searching and renting are separated, but to increase security, they could be accessed using two different passwords, so that if someone gains unlawful access, there is a limited amount of damage they could do.

5) Someone could potentially try to upload large csv files, so that if the file is raed it will crash the system. Before reading files, we could check the file size.